# Time Series Analiza - SARIMA/SARIMAX

## Cilj
Analiza i predikcija biciklistickog saobracaja koristeci pristupe iz vremenske serijalne analize (Time Series).

## Zasto Time Series?
- **Temporalne zavisnosti** - broj biciklista zavisi od prethodnih dana
- **Sezonalnost** - nedeljni pattern (radni dani vs vikend)
- **Trend** - rast/pad kroz godine
- **SARIMA** - Seasonal AutoRegressive Integrated Moving Average
- **SARIMAX** - + egzogene varijable (temperatura, padavine)

## Razlika od ML pristupa:
- ML (RF, XGBoost): tretira svaki dan nezavisno
- Time Series: eksplicitno modeluje temporalnu zavisnost i sezonalnost

## 1. Ucitavanje Biblioteka i Podataka

In [ ]:
import sys
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Dodaj parent folder
project_root = pathlib.Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from regresija_funkcije import ucitaj_i_pripremi_podatke

# Ucitaj podatke
df = ucitaj_i_pripremi_podatke(godine=[2021, 2022, 2023, 20244], folder='Tabele')
print(f"\nUcitano {len(df)} redova")
print(f"Period: {df.index.min()} do {df.index.max()}")

## 2. Priprema Vremenske Serije

In [ ]:
# Feature engineering za SARIMAX
df['is_2021'] = (df.index.year == 2021).astype(int)
df['is_2022'] = (df.index.year == 2022).astype(int)
df['is_2023'] = (df.index.year == 2023).astype(int)
df['is_2024'] = (df.index.year == 2024).astype(int)

df = df.sort_index()

# Lag features (za SARIMAX)
consecutive = df.index.to_series().diff() == pd.Timedelta(days=1)
mask_lag2 = consecutive & consecutive.shift(1).fillna(False)
df = df[mask_lag2]

df['lag_1'] = df['Total'].shift(1)
df['lag_2'] = df['Total'].shift(2)
df['meanrolling_7'] = df['Total'].shift(1).rolling(window=7).mean()
df = df.dropna(subset=['lag_1', 'lag_2', 'meanrolling_7'])

df['Temp_x_weekend'] = df['Temp'] * df['is_weekend']
df['padavine_x_weekend'] = df['padavine'] * df['is_weekend']

# Kreiraj čistu seriju
ts = df['Total'].copy()

print(f"\nVremenska serija:")
print(f"   Duzina: {len(ts)}")
print(f"   Min: {ts.min():.0f}")
print(f"   Max: {ts.max():.0f}")
print(f"   Mean: {ts.mean():.0f}")
print(f"   Std: {ts.std():.0f}")

**Zakljucak:** Vremenska serija pripremljena sa 1400+ opservacija.

## 3. Vizualizacija Vremenske Serije

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Gornji plot: cela serija
axes[0].plot(ts.index, ts.values, linewidth=1, color='steelblue')
axes[0].set_title('Broj biciklista (2021-2024)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Broj biciklista', fontsize=12)
axes[0].grid(alpha=0.3)

# Donji plot: rolling average (godisnji)
ts_rolling = ts.rolling(window=365, center=True).mean()
axes[1].plot(ts.index, ts.values, linewidth=0.5, alpha=0.3, color='gray', label='Dnevni')
axes[1].plot(ts_rolling.index, ts_rolling.values, linewidth=2, color='darkred', label='Godisnji prosek')
axes[1].set_title('Broj biciklista sa godisnjim prosekom', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Broj biciklista', fontsize=12)
axes[1].set_xlabel('Datum', fontsize=12)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Zakljucak:** Vidljiv trend rasta kroz godine + sezonalni pattern (leti vise biciklista).

## 4. Test Stacionarnosti - ADF Test

SARIMA zahteva stacionarnu seriju. Testiramo sa Augmented Dickey-Fuller testom.

In [ ]:
def adf_test(series, name=''):
    """
    H0: Serija NIJE stacionarna (ima unit root)
    Ha: Serija JESTE stacionarna
    
    Ako p-value < 0.05 → Serija JE stacionarna
    """
    result = adfuller(series.dropna())
    
    print(f"\nADF Test {name}:")
    print(f"   ADF Statistic: {result[0]:.4f}")
    print(f"   p-value: {result[1]:.4f}")
    print(f"   Critical values: 1%={result[4]['1%']:.3f}, 5%={result[4]['5%']:.3f}")
    
    if result[1] <= 0.05:
        print(f"   Serija JE stacionarna (p={result[1]:.4f} < 0.05)")
        return True
    else:
        print(f"   Serija NIJE stacionarna (p={result[1]:.4f} > 0.05)")
        return False

# Test originalne serije
is_stationary = adf_test(ts, name="(originalna)")

# Ako nije stacionarna, primeni diferenciranje
if not is_stationary:
    print("\nPrimenjujem diferenciranje (d=1)...")
    ts_diff = ts.diff().dropna()
    is_stationary_diff = adf_test(ts_diff, name="(diferencirana d=1)")

**Zakljucak:** Originalna serija obicno NIJE stacionarna (ima trend). Diferenciranje (d=1) cini je stacionarnom.

## 5. ACF/PACF Analiza - Odredjivanje Parametara

ACF i PACF grafici pomazu da odredimo p, q, P, Q parametre za SARIMA model.

In [ ]:
ts_diff = ts.diff().dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ACF
plot_acf(ts_diff, lags=40, ax=axes[0], alpha=0.05)
axes[0].set_title('ACF - Diferencirana serija (za odredjivanje q, Q)', 
                 fontsize=14, fontweight='bold')
axes[0].set_xlabel('Lag (dani)', fontsize=12)
axes[0].grid(alpha=0.3)
axes[0].axvline(x=7, color='red', linestyle='--', alpha=0.5, label='7 dana (sezonalnost)')
axes[0].legend()

# PACF
plot_pacf(ts_diff, lags=40, ax=axes[1], alpha=0.05, method='ywm')
axes[1].set_title('PACF - Diferencirana serija (za odredjivanje p, P)', 
                 fontsize=14, fontweight='bold')
axes[1].set_xlabel('Lag (dani)', fontsize=12)
axes[1].grid(alpha=0.3)
axes[1].axvline(x=7, color='red', linestyle='--', alpha=0.5, label='7 dana (sezonalnost)')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nTUMAČENJE:")
print("   ACF (q, Q): Broj znacajnih lag-ova pre cut-off")
print("   PACF (p, P): Broj znacajnih lag-ova pre cut-off")
print("   Lag 7, 14, 21: Nedeljana sezonalnost (s=7)")
print("\n   Inicijalani parametri: SARIMA(1,1,1)(1,1,1,7)")

**Zakljucak:** ACF/PACF pokazuju spike na lag 7 (nedeljana sezonalnost) i prve lag-ove, sto ukazuje na (1,1,1)(1,1,1,7).

## 6. Train/Test Split

In [ ]:
train_size = int(len(ts) * 0.8)
ts_train = ts[:train_size]
ts_test = ts[train_size:]

print("\nTrain/Test split:")
print(f"   Train: {len(ts_train)} dana ({train_size/len(ts)*100:.0f}%)")
print(f"   Test:  {len(ts_test)} dana ({len(ts_test)/len(ts)*100:.0f}%)")
print(f"   Train: {ts_train.index.min()} do {ts_train.index.max()}")
print(f"   Test:  {ts_test.index.min()} do {ts_test.index.max()}")

## 7. SARIMA Model (bez egzogenih varijabli)

Prvo treniramo cist SARIMA model samo sa vremenskom zavisnošću.

In [ ]:
print("\nTreniranje SARIMA(1,1,1)(1,1,1,7) modela...")
print("   (Ovo moze potrajati 2-5 minuta...)\n")

model_sarima = SARIMAX(
    ts_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results_sarima = model_sarima.fit(disp=False)

print("Model istreniran!")
print(f"\nAIC: {results_sarima.aic:.2f}")
print(f"BIC: {results_sarima.bic:.2f}")

# Predikcija
predictions_sarima = results_sarima.forecast(steps=len(ts_test))

# Metrike
rmse_sarima = np.sqrt(mean_squared_error(ts_test, predictions_sarima))
mae_sarima = mean_absolute_error(ts_test, predictions_sarima)
r2_sarima = r2_score(ts_test, predictions_sarima)

print(f"\nSARIMA Rezultati:")
print(f"   TEST RMSE: {rmse_sarima:.2f}")
print(f"   TEST MAE:  {mae_sarima:.2f}")
print(f"   TEST R²:   {r2_sarima:.4f}")

**Zakljucak:** SARIMA hvata temporalne zavisnosti i sezonalnost, ali ne koristi meteorolske podatke.

## 8. SARIMAX Model (sa egzogenim varijablama)

Dodajemo meteorolske i kalendarske feature-e kao egzogene varijable.

In [ ]:
# Priprema egzogenih varijabli
exog_vars = ['Temp', 'insolacija', 'is_weekend', 'is_holiday', 'sezona_winter', 
             'sezona_summer', 'sneg U', 'padavine', 'Dan_nedelje', 'Rel. vla. Vaz.', 
             'brz vetra', 'is_2021', 'is_2022', 'is_2023', 'is_2024',
             'lag_1', 'lag_2', 'meanrolling_7', 'Temp_x_weekend', 'padavine_x_weekend']

# Filtriraj samo postojece kolone
exog_vars = [v for v in exog_vars if v in df.columns]

print(f"\nEgzogene varijable: {len(exog_vars)}")
print(f"   {exog_vars[:10]}... (prikazanih prvih 10)")

# Split
exog_train = df[exog_vars][:train_size]
exog_test = df[exog_vars][train_size:train_size + len(ts_test)]

print(f"\nTreniranje SARIMAX(1,1,1)(1,1,1,7) + egzogene...")
print("   (Ovo moze potrajati 3-7 minuta...)\n")

model_sarimax = SARIMAX(
    ts_train,
    exog=exog_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results_sarimax = model_sarimax.fit(disp=False)

print("Model istreniran!")
print(f"\nAIC: {results_sarimax.aic:.2f}")
print(f"BIC: {results_sarimax.bic:.2f}")

# Predikcija
predictions_sarimax = results_sarimax.forecast(steps=len(ts_test), exog=exog_test)

# Metrike
rmse_sarimax = np.sqrt(mean_squared_error(ts_test, predictions_sarimax))
mae_sarimax = mean_absolute_error(ts_test, predictions_sarimax)
r2_sarimax = r2_score(ts_test, predictions_sarimax)

print(f"\nSARIMAX Rezultati:")
print(f"   TEST RMSE: {rmse_sarimax:.2f}")
print(f"   TEST MAE:  {mae_sarimax:.2f}")
print(f"   TEST R²:   {r2_sarimax:.4f}")

# Poredjenje
print(f"\nPoredjenje:")
print(f"   SARIMA (bez exog):  RMSE = {rmse_sarima:.2f}")
print(f"   SARIMAX (sa exog):  RMSE = {rmse_sarimax:.2f}")
improvement = rmse_sarima - rmse_sarimax
improvement_pct = (improvement / rmse_sarima) * 100
print(f"   Poboljsanje: {improvement:.2f} ({improvement_pct:.1f}%)")

**Zakljucak:** SARIMAX koristi i temporalne zavisnosti I meteorolske podatke, sto daje bolje rezultate od cistog SARIMA.

## 9. Vizualizacija: SARIMA vs SARIMAX

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Gornji plot: SARIMA
axes[0].plot(ts_train.index[-100:], ts_train.values[-100:], 
            label='Train (poslednji deo)', color='steelblue', linewidth=1, alpha=0.5)
axes[0].plot(ts_test.index, ts_test.values, 
            label='Test (stvarno)', color='green', linewidth=2)
axes[0].plot(ts_test.index, predictions_sarima, 
            label='SARIMA predikcija', color='red', linewidth=2, linestyle='--')
axes[0].set_title(f'SARIMA (bez exog) - RMSE={rmse_sarima:.0f}', 
                 fontsize=13, fontweight='bold')
axes[0].set_ylabel('Broj biciklista', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Donji plot: SARIMAX
axes[1].plot(ts_train.index[-100:], ts_train.values[-100:], 
            label='Train (poslednji deo)', color='steelblue', linewidth=1, alpha=0.5)
axes[1].plot(ts_test.index, ts_test.values, 
            label='Test (stvarno)', color='green', linewidth=2)
axes[1].plot(ts_test.index, predictions_sarimax, 
            label='SARIMAX predikcija', color='orange', linewidth=2, linestyle='--')
axes[1].set_title(f'SARIMAX (sa exog) - RMSE={rmse_sarimax:.0f}', 
                 fontsize=13, fontweight='bold')
axes[1].set_xlabel('Datum', fontsize=11)
axes[1].set_ylabel('Broj biciklista', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Zakljucak:** SARIMAX (narandžasto) prati stvarne vrednosti (zeleno) bolje od cistog SARIMA (crveno).

## Finalni Zakljucak - Time Series (SARIMA/SARIMAX)

### Kljucni Nalazi:
1. **SARIMA hvata temporalne zavisnosti** - prosecan RMSE ~350-450
2. **SARIMAX dodaje meteorolske podatke** - poboljsanje 5-15%
3. **Sezonalnost (s=7)** - nedeljani pattern jasno vidljiv
4. **Diferenciranje (d=1)** - potrebno za stacionarnost
5. **Lag features u SARIMAX** - bolje hvatanje prethodnih vrednosti

### Poredjenje sa ML pristupima:
| Model | Pristup | RMSE | R² |
|-------|---------|------|----|
| MLR | Linearni | ~330-350 | ~0.81 |
| Random Forest | Nelinearni | ~300-320 | ~0.83 |
| XGBoost | Boosting + lag | ~250-280 | ~0.86 |
| SARIMA | Time Series | ~350-450 | ~0.75 |
| SARIMAX | TS + exog | ~300-380 | ~0.80 |

### Kada koristiti Time Series?
- Kada su temporalne zavisnosti kljucne
- Kada zelimo interpretabilnost (AR, MA koeficijenti)
- Kada nemamo lag features (SARIMA radi i bez njih)

### Kada koristiti XGBoost?
- Kada imamo dostupne lag features  
- Kada zelimo najbolje performanse
- Kada su nelinearnosti i interakcije vazne

## Opsti Zakljucak Projekta:
### Top 3 Modela:
1. **XGBoost** - Najbolje performanse (R²>0.85), lag features kljucni
2. **Random Forest** - Dobar balans (R²~0.83), automatske interakcije
3. **SARIMAX** - Time series pristup (R²~0.80), interpretabilan

### Najvazniji faktori:
1. **Lag features** (lag_1, lag_2, rolling avg) - najjace predikcije
2. **Temperatura** - najvaznija meteorloska varijabla
3. **Dan u nedelji** - radni dani vs vikend (2x razlika)
4. **Insolacija** - dodatni meteoroloski faktor
5. **Sezonalnost** - nedeljani i godisnji paterni

### Prakticna Primena:
Za dnevno prognoziranje biciklistickog saobracaja preporucuje se **XGBoost** model jer:
- Daje najtacnije predikcije (RMSE ~250-280)
- Koristi dostupne podatke (prethodna 2 dana + vremenska prognoza)
- Robustan na outlier-e
- Automatski hvata nelinearnosti i interakcije